# **Analisis de Mejor alternativa de Producto** #

## Hecho por ##

Rene Zapata Hernandez

Camilo Andres Mendez Cogollo

Estid Esteban Vásquez


Entrega anterior

In [ ]:
pip install requests beautifulsoup4

In [ ]:
#PRUEBA DE CONEXION
import requests
from bs4 import BeautifulSoup

# URL del producto
url = "https://www.ktronix.com/iphone-duo-2tb-5g-blanco-estrella/p/195951407074"

# Encabezados para realizar la solicitud
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/141.0 Safari/537.36"
    )
}

# Realizar solicitud HTTP
response = requests.get(url, headers=headers, timeout=15)

print("Código de respuesta:", response.status_code)

if response.status_code == 200:

    # Procesar HTML
    soup = BeautifulSoup(response.text, "html.parser")

    # Obtener título de la página
    titulo = soup.title.get_text(strip=True)

    print("\n--- INFORMACIÓN DEL PRODUCTO ---")
    print("Título:", titulo)

else:
    print("No fue posible acceder a la página.")

DESARROLLO WEB SCRAPING
Web Scraping Básico con BeautifulSoup
Comercio: Ktronix
Producto: iPhone Duo 2TB 5G Blanco Estrella

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import csv
import re

def extraer_datos_producto(url: str) -> dict:
    # 1. encabezados HTTP Para simular un navegador real
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "es-CO,es;q=0.9,en;q=0.8"
    }

    print(f" Conectando a: {url}")
    response = requests.get(url, headers=headers, timeout=15)

    # Validar respuesta del servidor (HTTP 200 OK)
    if response.status_code != 200:
        raise Exception(f"Error al cargar la página. Código de estado: {response.status_code}")

    print("[*] Código HTML recibido. Parseando con BeautifulSoup...")
    soup = BeautifulSoup(response.text, "html.parser")

    # 2. Extracción de datos principales

    # Nombre del producto
    titulo_tag = soup.find("h1", class_="js-main-title")
    nombre = titulo_tag.get_text(strip=True) if titulo_tag else "No disponible"

    # SKU / Código EAN
    sku_tag = soup.find("span", class_="js-ean-pdp")
    sku = sku_tag.get_text(strip=True) if sku_tag else "No disponible"

    # Precio actual
    precio_tag = soup.find(id="js-original_price")
    precio_actual = "No disponible"
    if precio_tag:
        # Remover etiquetas internas accesorias como el span 'Hoy'
        etiqueta_hoy = precio_tag.find(class_="font-title--product-price__today")
        if etiqueta_hoy:
            etiqueta_hoy.decompose()
        precio_actual = precio_tag.get_text(strip=True)

    # Precio base / antes
    precio_base_tag = soup.find(class_="before-price__basePrice")
    precio_base = precio_base_tag.get_text(strip=True) if precio_base_tag else precio_actual

    # Moneda (extraída de metadatos Schema.org)
    moneda_tag = soup.find("meta", itemprop="priceCurrency")
    moneda = moneda_tag["content"] if moneda_tag and moneda_tag.has_attr("content") else "COP"

    # Categorías / Breadcrumbs
    categorias = []
    breadcrumb_wrapper = soup.find(class_=re.compile(r"breadcrumb", re.I))
    if breadcrumb_wrapper:
        for enlace in breadcrumb_wrapper.find_all("a"):
            texto = enlace.get_text(strip=True)
            if texto and texto not in categorias:
                categorias.append(texto)

    imagen_url = "No disponible"
    for img in soup.find_all("img"):
        src = img.get("src") or img.get("data-src") or ""
        if "products" in src and sku in src:
            imagen_url = src.split("?")[0]
            break

    # Especificaciones técnicas completas (tabla de atributos)
    especificaciones = {}
    items_specs = soup.find_all(class_="new-container__table__classifications___type__item")
    for item in items_specs:
        caracteristica = item.find(class_="new-container__table__classifications___type__item_feature")
        valor = item.find(class_="new-container__table__classifications___type__item_result")
        if caracteristica and valor:
            k = caracteristica.get_text(strip=True)
            v = re.sub(r"\s+", " ", valor.get_text(strip=True))
            if k and v:
                especificaciones[k] = v

    # Descripción comercial
    descripcion = "No disponible"
    for script in soup.find_all("script"):
        if script.string and "richSnippet(" in script.string:
            match = re.search(r'"description":\s*"([^"]+)"', script.string)
            if match:
                descripcion = match.group(1).encode("utf-8").decode("unicode_escape")
            break

    return {
        "url": url,
        "nombre": nombre,
        "sku": sku,
        "precio_actual": precio_actual,
        "precio_base": precio_base,
        "moneda": moneda,
        "categorias": " > ".join(categorias) if categorias else "No disponible",
        "imagen_url": imagen_url,
        "descripcion": descripcion,
        "especificaciones_principales": especificaciones
    }


def guardar_archivos(datos: dict, nombre_archivo: str = "producto_ktronix"):
    # Guardar en JSON estructurado
    with open(f"{nombre_archivo}.json", "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=4)
    print(f"OK Archivo JSON generado: {nombre_archivo}.json")


if __name__ == "__main__":
    url_producto = "https://www.ktronix.com/iphone-duo-2tb-5g-blanco-estrella/p/195951407074"


    try:
        datos = extraer_datos_producto(url_producto)

        print("\n--- DATOS EXTRAÍDOS ---")
        print(f"Producto     : {datos['nombre']}")
        print(f"SKU / Código : {datos['sku']}")
        print(f"Precio Actual: {datos['precio_actual']} {datos['moneda']}")
        print(f"Categorías   : {datos['categorias']}")
        print(f"Total Specs  : {len(datos['especificaciones_principales'])} especificaciones técnicas")

        print("\n" + "=" * 60)
        guardar_archivos(datos)
        print("=" * 60)
        print("Proceso completado exitosamente.")

    except Exception as e:
        print(f"[ERROR] Error durante la ejecución: {e}")

datataframe para mostrar los datos extraidos

In [ ]:
import pandas as pd

# Crear un diccionario con los datos principales solicitados
datos_principales = {
    "Campo": [
        "Producto",
        "SKU / Código",
        "Precio Actual",
        "Categorías",
        "Total Specs"
    ],
    "Valor": [
        datos.get("nombre", "No disponible"),
        datos.get("sku", "No disponible"),
        f"{datos.get('precio_actual', 'No disponible')} {datos.get('moneda', '')}".strip(),
        datos.get("categorias", "No disponible"),
        f"{len(datos.get('especificaciones_principales', {}))} especificaciones técnicas"
    ]
}

# Crear el DataFrame
df_producto = pd.DataFrame(datos_principales)

# Mostrar el DataFrame
df_producto